# Imbalanced Learning version 3

Hyper-parameter tuning for each model and using 3 classes (agregating 2 and 3 into 2)

## Data Preparation

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter
from collections import defaultdict
from sklearn.preprocessing import StandardScaler

In [2]:
#df = pd.read_csv('cmi_internet_cleaned.csv')
df = pd.read_csv('../dataset/cmi_internet_cleaned.csv')
#df

In [3]:
X = df.drop(columns=['sii'])
y = df['sii'].astype(int)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
print("Variables used:")
print(num_cols)
print("Shape:", X.shape)

Variables used:
['Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score', 'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference', 'Physical-Diastolic_BP', 'Fitness_Endurance-Max_Stage', 'Physical-HeartRate', 'Physical-Systolic_BP', 'FGC-FGC_CU', 'FGC-FGC_GSND', 'FGC-FGC_GSD', 'FGC-FGC_PU', 'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL', 'BIA-BIA_DEE', 'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMR', 'BIA-BIA_ECW', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_ICW', 'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM', 'PAQ_Total', 'SDS-SDS_Total_T', 'PreInt_EduHx-computerinternet_hoursday', 'Fitness_Endurance-Time']
Shape: (7839, 32)


In [4]:
scaler = StandardScaler() # very important for KNN

X = scaler.fit_transform(X)

X = pd.DataFrame(
    X,
    columns=num_cols,
    index=df.index
)

#X

## 3-class classification problem

In [5]:
# The four classes of sii are imbalanced:
ctr = Counter(y)
ctr

Counter({0: 5476, 1: 1444, 2: 846, 3: 73})

In [6]:
5476/(5476+1444+846+73)*100, 1444/(5476+1444+846+73)*100, (846+73)/(5476+1444+846+73)*100

(69.85584896032657, 18.420716928179615, 11.723434111493813)

We will turn it into a 3 class classification problem with imbalanced classes, by grouping the classes into 0 (None), 1 (Mild), and the classes 2 (Moderate) and 3 (Severe) now together into 2 (High)

In [7]:
y2 = y.copy()
y2[y2 > 1] = 2 
y2

0       2
1       0
2       0
3       1
4       0
       ..
7834    0
7835    1
7836    0
7837    2
7838    0
Name: sii, Length: 7839, dtype: int64

In [8]:
Counter(y2)

Counter({0: 5476, 1: 1444, 2: 919})

## Data Partitioning

In [9]:
from sklearn.model_selection import train_test_split, cross_val_score 

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y2, test_size=0.3, random_state=100, stratify=y2)

In [11]:
# Records of each class in train and test datasets keep the proportion of the original dataset thanks to 'stratify':
np.unique(y_train, return_counts=True), np.unique(y_test, return_counts=True)

((array([0, 1, 2]), array([3833, 1011,  643])),
 (array([0, 1, 2]), array([1643,  433,  276])))

## Basic Classification

In [12]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
#from sklearn.metrics import roc_curve, auc, roc_auc_score

In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold

In [15]:
import warnings
warnings.simplefilter("ignore")

In [16]:
clf = DummyClassifier() # classifies everything as the majority class (class 0 in this case)
clf.fit(X_train, y_train)

y_pred0 = clf.predict(X_test)
print(classification_report(y_test, y_pred0))

              precision    recall  f1-score   support

           0       0.70      1.00      0.82      1643
           1       0.00      0.00      0.00       433
           2       0.00      0.00      0.00       276

    accuracy                           0.70      2352
   macro avg       0.23      0.33      0.27      2352
weighted avg       0.49      0.70      0.57      2352



In [21]:
# Hyper-parameter tuning for Decision Tree Classifier
# Default: gini criterion, no max depth, min_samples_split=2, min_samples_leaf=1, no class weights

param_list = {
    'max_depth': [None] + list(np.arange(2, 20)),
    'min_samples_split': [2, 5, 10, 20, 30, 50],
    'min_samples_leaf': [1, 5, 10, 20, 30, 50],
    'criterion': ['gini', 'entropy']
}

def doDT(X,y):
    random_search = RandomizedSearchCV(
        DecisionTreeClassifier(),
        param_distributions=param_list,
        scoring='balanced_accuracy',
        cv=RepeatedStratifiedKFold(random_state=0),
        n_jobs=-1,
        refit=True,
        n_iter=200,
        #verbose=2
    )

    random_search.fit(X, y)
    clf = random_search.best_estimator_

    print(random_search.best_params_, random_search.best_score_)
    clf.fit(X, y)

    y_pred0 = clf.predict(X_test)
    print(classification_report(y_test, y_pred0))

In [18]:
doDT(X_train,y_train)

{'min_samples_split': 10, 'min_samples_leaf': 10, 'max_depth': 14, 'criterion': 'gini'} 0.3947019786672518
              precision    recall  f1-score   support

           0       0.74      0.83      0.78      1643
           1       0.27      0.21      0.24       433
           2       0.26      0.17      0.21       276

    accuracy                           0.64      2352
   macro avg       0.42      0.40      0.41      2352
weighted avg       0.60      0.64      0.61      2352



In [19]:
# Hyper-parameter tuning for KNN Classifier
# Default: 5 neighbours, minkowski distance, uniform weights

param_grid = {
    "n_neighbors": [e for e in range(1, 50) if e % 2 != 0],
    "weights": ["uniform", "distance"],
    "metric": ["minkowski", "cityblock"],
}

def doKNN(X,y):
    random_search = RandomizedSearchCV(
        KNeighborsClassifier(),
        param_distributions=param_grid,
        scoring='balanced_accuracy',
        cv=RepeatedStratifiedKFold(random_state=0),
        n_jobs=-1,
        refit=True,
        n_iter=200,
        #verbose=2
    )

    random_search.fit(X, y)
    clf = random_search.best_estimator_

    print(random_search.best_params_, random_search.best_score_)
    clf.fit(X, y)

    y_pred0 = clf.predict(X_test)
    print(classification_report(y_test, y_pred0))

In [20]:
doKNN(X_train,y_train)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'minkowski'} 0.3772089569126637
              precision    recall  f1-score   support

           0       0.73      0.78      0.75      1643
           1       0.25      0.22      0.23       433
           2       0.21      0.17      0.19       276

    accuracy                           0.60      2352
   macro avg       0.40      0.39      0.39      2352
weighted avg       0.58      0.60      0.59      2352



## Undersampling

In [22]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.under_sampling import CondensedNearestNeighbour
from imblearn.under_sampling import TomekLinks
from imblearn.under_sampling import EditedNearestNeighbours

### RandomUnderSampler

In [23]:
rus = RandomUnderSampler(random_state=42)
X_res, y_res = rus.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 643, 1: 643, 2: 643})


In [24]:
doDT(X_res,y_res)

{'min_samples_split': 30, 'min_samples_leaf': 30, 'max_depth': 2, 'criterion': 'entropy'} 0.4622726905684755
              precision    recall  f1-score   support

           0       0.80      0.60      0.69      1643
           1       0.25      0.17      0.21       433
           2       0.19      0.57      0.29       276

    accuracy                           0.52      2352
   macro avg       0.42      0.45      0.39      2352
weighted avg       0.63      0.52      0.55      2352



In [25]:
doKNN(X_res,y_res)

{'weights': 'distance', 'n_neighbors': 43, 'metric': 'minkowski'} 0.43779635012919904
              precision    recall  f1-score   support

           0       0.77      0.75      0.76      1643
           1       0.28      0.16      0.20       433
           2       0.25      0.47      0.33       276

    accuracy                           0.61      2352
   macro avg       0.43      0.46      0.43      2352
weighted avg       0.62      0.61      0.61      2352



### CondensedNearestNeighbour

In [26]:
cnn = CondensedNearestNeighbour(random_state=42, n_jobs=10)
X_res, y_res = cnn.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 1076, 2: 643, 1: 566})


In [27]:
doDT(X_res,y_res)

{'min_samples_split': 50, 'min_samples_leaf': 1, 'max_depth': 4, 'criterion': 'gini'} 0.3975209597207195
              precision    recall  f1-score   support

           0       0.74      0.87      0.80      1643
           1       0.29      0.03      0.06       433
           2       0.25      0.33      0.28       276

    accuracy                           0.66      2352
   macro avg       0.43      0.41      0.38      2352
weighted avg       0.60      0.66      0.61      2352



In [28]:
doKNN(X_res,y_res)

{'weights': 'distance', 'n_neighbors': 25, 'metric': 'cityblock'} 0.363299415140323
              precision    recall  f1-score   support

           0       0.73      0.94      0.82      1643
           1       0.34      0.05      0.08       433
           2       0.29      0.18      0.22       276

    accuracy                           0.69      2352
   macro avg       0.45      0.39      0.38      2352
weighted avg       0.60      0.69      0.61      2352



### Tomek Links

In [29]:
tl = TomekLinks()
X_res, y_res = tl.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 3625, 1: 829, 2: 643})


In [30]:
doDT(X_res,y_res)

{'min_samples_split': 30, 'min_samples_leaf': 5, 'max_depth': 16, 'criterion': 'gini'} 0.40093892461415725
              precision    recall  f1-score   support

           0       0.74      0.84      0.79      1643
           1       0.28      0.19      0.23       433
           2       0.28      0.19      0.23       276

    accuracy                           0.65      2352
   macro avg       0.43      0.41      0.41      2352
weighted avg       0.60      0.65      0.62      2352



In [31]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'minkowski'} 0.4065769179668786
              precision    recall  f1-score   support

           0       0.73      0.79      0.76      1643
           1       0.26      0.19      0.22       433
           2       0.20      0.18      0.19       276

    accuracy                           0.61      2352
   macro avg       0.39      0.39      0.39      2352
weighted avg       0.58      0.61      0.59      2352



### Edited Nearest Neighbors

In [32]:
enn = EditedNearestNeighbours()
X_res, y_res = enn.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 1968, 2: 643, 1: 7})


In [33]:
doDT(X_res,y_res)

{'min_samples_split': 20, 'min_samples_leaf': 30, 'max_depth': 16, 'criterion': 'entropy'} 0.46558597788328165
              precision    recall  f1-score   support

           0       0.76      0.80      0.78      1643
           1       0.00      0.00      0.00       433
           2       0.24      0.53      0.33       276

    accuracy                           0.62      2352
   macro avg       0.33      0.44      0.37      2352
weighted avg       0.56      0.62      0.58      2352



In [34]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.49574783470202033
              precision    recall  f1-score   support

           0       0.74      0.84      0.78      1643
           1       0.50      0.01      0.02       433
           2       0.20      0.36      0.26       276

    accuracy                           0.63      2352
   macro avg       0.48      0.40      0.35      2352
weighted avg       0.63      0.63      0.58      2352



### Cluster Centroids

In [35]:
from sklearn.cluster import MiniBatchKMeans
from imblearn.under_sampling import ClusterCentroids

In [36]:
cc = ClusterCentroids(estimator=MiniBatchKMeans(n_init=1, random_state=0), random_state=42)

X_res, y_res = cc.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({0: 643, 1: 643, 2: 643})


In [37]:
doDT(X_res,y_res)

{'min_samples_split': 50, 'min_samples_leaf': 1, 'max_depth': 18, 'criterion': 'entropy'} 0.5094981427648578
              precision    recall  f1-score   support

           0       0.78      0.24      0.37      1643
           1       0.21      0.28      0.24       433
           2       0.14      0.63      0.23       276

    accuracy                           0.29      2352
   macro avg       0.38      0.38      0.28      2352
weighted avg       0.60      0.29      0.33      2352



In [38]:
doKNN(X_res,y_res)

{'weights': 'distance', 'n_neighbors': 47, 'metric': 'cityblock'} 0.48752099483204125
              precision    recall  f1-score   support

           0       0.74      0.77      0.75      1643
           1       0.31      0.05      0.09       433
           2       0.22      0.46      0.29       276

    accuracy                           0.60      2352
   macro avg       0.42      0.42      0.38      2352
weighted avg       0.60      0.60      0.58      2352



## Oversampling

In [39]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import ADASYN

### RandomOverSampler

In [40]:
ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({1: 3833, 0: 3833, 2: 3833})


In [41]:
doDT(X_res,y_res)

{'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'entropy'} 0.8832242877713515
              precision    recall  f1-score   support

           0       0.75      0.72      0.74      1643
           1       0.24      0.25      0.25       433
           2       0.21      0.24      0.22       276

    accuracy                           0.58      2352
   macro avg       0.40      0.41      0.40      2352
weighted avg       0.59      0.58      0.59      2352



In [42]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.9104096981332895
              precision    recall  f1-score   support

           0       0.73      0.78      0.76      1643
           1       0.22      0.21      0.21       433
           2       0.20      0.14      0.16       276

    accuracy                           0.60      2352
   macro avg       0.38      0.38      0.38      2352
weighted avg       0.58      0.60      0.59      2352



### SMOTE

In [43]:
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({1: 3833, 0: 3833, 2: 3833})


In [44]:
doDT(X_res,y_res)

{'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 19, 'criterion': 'entropy'} 0.6855901906652005
              precision    recall  f1-score   support

           0       0.75      0.66      0.70      1643
           1       0.24      0.30      0.27       433
           2       0.17      0.24      0.20       276

    accuracy                           0.54      2352
   macro avg       0.39      0.40      0.39      2352
weighted avg       0.59      0.54      0.56      2352



In [45]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.8832593843294378
              precision    recall  f1-score   support

           0       0.75      0.68      0.71      1643
           1       0.23      0.29      0.26       433
           2       0.20      0.24      0.22       276

    accuracy                           0.55      2352
   macro avg       0.39      0.40      0.40      2352
weighted avg       0.59      0.55      0.57      2352



### ADASYN

In [46]:
ada = ADASYN(random_state=42)
X_res, y_res = ada.fit_resample(X_train, y_train)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({2: 3930, 0: 3833, 1: 3715})


In [47]:
doDT(X_res,y_res)

{'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'gini'} 0.6719493726289102
              precision    recall  f1-score   support

           0       0.73      0.67      0.70      1643
           1       0.22      0.24      0.23       433
           2       0.15      0.20      0.18       276

    accuracy                           0.54      2352
   macro avg       0.37      0.37      0.37      2352
weighted avg       0.57      0.54      0.55      2352



In [48]:
doKNN(X_res,y_res)

{'weights': 'uniform', 'n_neighbors': 1, 'metric': 'cityblock'} 0.8857318407348548
              precision    recall  f1-score   support

           0       0.75      0.68      0.72      1643
           1       0.22      0.28      0.25       433
           2       0.20      0.24      0.22       276

    accuracy                           0.56      2352
   macro avg       0.39      0.40      0.39      2352
weighted avg       0.59      0.56      0.57      2352



## Balancing at the Algorithm Level

### Class Weight

In [49]:
param_list = {
    'max_depth': [None] + list(np.arange(2, 20)),
    'min_samples_split': [2, 5, 10, 20, 30, 50],
    'min_samples_leaf': [1, 5, 10, 20, 30, 50],
    'class_weight': [{0:1, 1:2, 2:3, 3:4}, {0:1, 1:5, 2:10, 3:20}, {0:1, 1:10, 2:25, 3:50}],
    'criterion': ['gini', 'entropy']
}

doDT(X_train,y_train)

{'min_samples_split': 30, 'min_samples_leaf': 1, 'max_depth': 2, 'criterion': 'gini', 'class_weight': {0: 1, 1: 5, 2: 10, 3: 20}} 0.4530798070156694
              precision    recall  f1-score   support

           0       0.80      0.59      0.68      1643
           1       0.00      0.00      0.00       433
           2       0.18      0.73      0.28       276

    accuracy                           0.50      2352
   macro avg       0.33      0.44      0.32      2352
weighted avg       0.58      0.50      0.51      2352

